<h1>Содержание<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Подготовка" data-toc-modified-id="Подготовка-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Подготовка</a></span><ul class="toc-item"><li><span><a href="#Импортируем-необходимые-библиотеки-и-загрузим-данные" data-toc-modified-id="Импортируем-необходимые-библиотеки-и-загрузим-данные-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Импортируем необходимые библиотеки и загрузим данные</a></span></li><li><span><a href="#Так-как-у-нас-имеется-ограничение-по-ресурсам,-сбалансируем-наши-данные-по-целевому-признаку-а-также-сделаем-выборку-в-32000-строк." data-toc-modified-id="Так-как-у-нас-имеется-ограничение-по-ресурсам,-сбалансируем-наши-данные-по-целевому-признаку-а-также-сделаем-выборку-в-32000-строк.-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Так как у нас имеется ограничение по ресурсам, сбалансируем наши данные по целевому признаку а также сделаем выборку в 32000 строк.</a></span></li><li><span><a href="#Скачаем-библиотеки-для-обработки-языка-и-произведем-лемматизацию-текста" data-toc-modified-id="Скачаем-библиотеки-для-обработки-языка-и-произведем-лемматизацию-текста-1.3"><span class="toc-item-num">1.3&nbsp;&nbsp;</span>Скачаем библиотеки для обработки языка и произведем лемматизацию текста</a></span></li><li><span><a href="#Разделим-наши-подготовленные-данные-на-тренировочную-и-тестовую-выборки" data-toc-modified-id="Разделим-наши-подготовленные-данные-на-тренировочную-и-тестовую-выборки-1.4"><span class="toc-item-num">1.4&nbsp;&nbsp;</span>Разделим наши подготовленные данные на тренировочную и тестовую выборки</a></span></li></ul></li><li><span><a href="#Обучение" data-toc-modified-id="Обучение-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Обучение</a></span><ul class="toc-item"><li><span><a href="#Произведем-обучение-на-подготвленных-даных-на-2х-моделях:" data-toc-modified-id="Произведем-обучение-на-подготвленных-даных-на-2х-моделях:-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Произведем обучение на подготвленных даных на 2х моделях:</a></span></li></ul></li><li><span><a href="#Выводы" data-toc-modified-id="Выводы-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Выводы</a></span></li></ul></div>

# Проект для «Викишоп»

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию. 

Обучите модель классифицировать комментарии на позитивные и негативные. В вашем распоряжении набор данных с разметкой о токсичности правок.

Постройте модель со значением метрики качества *F1* не меньше 0.75. 

**План исследования**

1. Загрузка и подготовка данных.
2. Обучение разных моделей. 
3. Выводы.

**Описание данных**

Данные находятся в файле **`toxic_comments.csv`**. Столбец **`text`** в нём содержит текст комментария, а **`toxic`** — целевой признак.

## Подготовка

### Импортируем необходимые библиотеки и загрузим данные

In [1]:
import pandas as pd
import numpy as np

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier, Pool, cv
from sklearn.pipeline import make_pipeline, FeatureUnion, Pipeline
from sklearn.metrics import f1_score
from sklearn.feature_selection import SelectPercentile, chi2

from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack, csr_matrix
from gensim.models import Word2Vec

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/jovyan/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
data = pd.read_csv('/datasets/toxic_comments.csv', index_col=0)

In [3]:
data.head()

,text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 159292 entries, 0 to 159450
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    159292 non-null  object
 1   toxic   159292 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.6+ MB


In [5]:
data.isnull().sum()

text     0
toxic    0
dtype: int64

In [6]:
data['toxic'].value_counts()

0    143106
1     16186
Name: toxic, dtype: int64

In [7]:
data.duplicated().sum()

0

Необходимые библиотеки загружены.<br>
По загруженным данным у нас 159292 строк. Из них: <br>
   * 143106 - нетоксичные комментарии, <br>
   * 16186 - токсичные, дикий дисбаланс. Нетоксичных комментариев примерно в 9 раз больше. <br>
   
Дисбаланс необходимо будет учесть при подготовке данных.

### Так как у нас имеется ограничение по ресурсам, сбалансируем наши данные по целевому признаку а также сделаем выборку в 32000 строк.

In [9]:
total_samples = 32000

toxic_ratio = len(data[data['toxic'] == 1]) / len(data)        
nontoxic_ratio = len(data[data['toxic'] == 0]) / len(data)    

target_toxic = int(total_samples * toxic_ratio)    
target_nontoxic = total_samples - target_toxic      

sampled_toxic = data[data['toxic'] == 1].sample(n=target_toxic, random_state=42)
sampled_nontoxic = data[data['toxic'] == 0].sample(n=target_nontoxic, random_state=42)

data = pd.concat([sampled_toxic, sampled_nontoxic])
data = data.reset_index(drop=True)
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

### Скачаем библиотеки для обработки языка и произведем лемматизацию текста

In [10]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Очистка текста от пунктуации и цифр"""
    # Удаляем все символы кроме букв и пробелов
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Убираем лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {
        'J': wordnet.ADJ,   
        'N': wordnet.NOUN,    
        'V': wordnet.VERB, 
        'R': wordnet.ADV  
    }
    return tag_dict.get(tag, wordnet.NOUN)

def lemmatize_english(texts):
    results = []
    for text in tqdm(texts, desc='Лемматизация'):
        text = clean_text(text)
      
        words = text.split()
        
        #words = [w for w in words if w.lower() not in stop_words]
        
        lemm_words = []
        for word in words:
            try:
                pos = get_wordnet_pos(word.lower())
                lemm_words.append(lemmatizer.lemmatize(word.lower(), pos))
            except:
                lemm_words.append(lemmatizer.lemmatize(word.lower()))
        
        results.append(' '.join(lemm_words))
    return results

corpus = data['text'].astype(str).tolist()
data['lemm_text'] = lemmatize_english(corpus)

Лемматизация: 100%|██████████| 32000/32000 [03:23<00:00, 157.57it/s]


In [11]:
count_tf_idf = TfidfVectorizer(stop_words='english')

### Разделим наши подготовленные данные на тренировочную и тестовую выборки

In [12]:
X = data['lemm_text']
y = data['toxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.25, random_state = 42,
    shuffle = True, stratify = y)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True) 
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

## Обучение

### Произведем обучение на подготвленных даных на 2х моделях:
    - Логистическая регрессия
    - Катбуст классификатор

Для наших моделей выберем параметры, которые помогают оставить только информативные признаки и избавиться от шума, что улучшает качество модели и ускоряет обучение.

In [13]:
param_dist = {
    'tfidfvectorizer__max_features': [5000, 8000, 10000, 15000, 20000],
    'tfidfvectorizer__min_df': [1, 2, 3, 5],
    'tfidfvectorizer__max_df': [0.6, 0.7, 0.8, 0.9, 0.95],
    'tfidfvectorizer__sublinear_tf': [True, False],
    'tfidfvectorizer__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'logisticregression__C': [0.3, 0.5, 0.7, 1.0, 1.5, 2.0],
    'logisticregression__penalty': ['l1', 'l2'], 
    'logisticregression__solver': ['liblinear'] 
}

pipeline = make_pipeline(
    TfidfVectorizer(),
    SelectPercentile(chi2, percentile=50),
    LogisticRegression(
        max_iter=10000, 
        random_state=42,
        class_weight='balanced'  
    )
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rand = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=30,
    cv=cv, 
    scoring='f1',
    verbose=0,
    n_jobs=-1,
    random_state=42,
    refit=True 
)

rand.fit(X_train, y_train)

print(f'Лучший F1 на CV: {rand.best_score_:.4f}')

Лучший F1 на CV: 0.7398


In [14]:
X_train_df = pd.DataFrame({
    'text': X_train 
})

X_test_df = pd.DataFrame({
    'text': X_test
})

catboost_model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='F1',
    random_seed=42,
    task_type='CPU',
    verbose=0,
    early_stopping_rounds=50,
    text_features=['text'],
    auto_class_weights=None
)

param_dist = {
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'depth': [4, 6],
    'l2_leaf_reg': [1, 3, 5, 7],
}

random_search = RandomizedSearchCV(
    estimator=catboost_model,
    param_distributions=param_dist,
    n_iter=10,
    scoring='f1',
    cv=2,
    n_jobs=-1,
    verbose=0,
    random_state=42,
    return_train_score=True
)

random_search.fit(X_train_df, y_train)

print(f'Лучший F1 на CV: {random_search.best_score_:.4f}')

Лучший F1 на CV: 0.7510


## Выводы

Таким образом лучшую метрику на кросс валидации у нас показала модель CatBoost. Проверим показатель f1 на тестовой выборке.

In [15]:
y_pred = random_search.best_estimator_.predict(X_test_df)
f1_catboost_test = f1_score(y_test, y_pred)
f1_catboost_test.round(3)

0.791

В ходе эксперимента были обучены две модели классификации для определения токсичных комментариев: логистическая регрессия (LogisticRegression) и градиентный бустинг (CatBoostClassifier). Обе модели показали сопоставимо высокие результаты.

* Так как у нас имеется ограничение по ресурсам, сбалансировали наши данные по целевому признаку а также сделали выборку в 32000 строк
* Обе модели продемонстрировали высокое качество классификации (F1 > 0.73), что говорит об эффективности выбранного подхода с использованием TF-IDF векторизации текстов для модели логистической регрессии.
* Разница в качестве между моделями составляет 0.012 в пользу CatBoostClassifier, что является статистически незначимым на данной выборке.
* Учитывая, что логистическая регрессия обучается значительно быстрее градиентного бустинга, она является более предпочтительной для практического использования в данном случае, несмотря на лучший показатель второй модели.